# Phase 2: EDA and baseline forecasting

This notebook explores CA_1 recorded sales and evaluates three simple forecasts. Sales are a demand proxy: zero sales do not establish either zero demand or a stockout.

**Protocol:** train on January 1, 2014 through December 3, 2015 and validate on December 4–31, 2015. All 28 validation forecasts use one fixed origin at the end of December 3. These are preliminary validation results. No independent final test or demand-regime classification is completed here.

Run all cells in order. The input is the Phase 1 prepared CSV. Figures and result tables go into `phase2_outputs/` beside this notebook.

In [ ]:
from pathlib import Path
import gc
import json
import hashlib
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

OUTPUT_DIR = Path("phase2_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

print("python     :", sys.version.split()[0])
print("pandas     :", pd.__version__)
print("numpy      :", np.__version__)
print("matplotlib :", plt.matplotlib.__version__)
print("seaborn    :", sns.__version__)


## Load previous work 

In [ ]:
DATA_DIR = Path("../data")
df = pd.read_csv(DATA_DIR / "m5_ca1_2014_2015_prepared.csv.gz", parse_dates=["date"])
df.shape

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df["day_int"] = df["d"].str.replace("d_", "", regex=False).astype(int)
df = df.sort_values(["id", "date"]).reset_index(drop=True)
print("Date range:", df["date"].min().date(), "to", df["date"].max().date())
print("Day numbers run from", df["day_int"].min(), "to", df["day_int"].max())

In [ ]:
# Determine cutoff date for the final 28 days of 2015
split_date = df['date'].max() - pd.Timedelta(days=28)

# Create chronological train and validation splits
train_df = df[df['date'] <= split_date].copy()
val_df = df[df['date'] > split_date].copy()

print(f"Train Set Horizon: {train_df['date'].min().strftime('%Y-%m-%d')} to {train_df['date'].max().strftime('%Y-%m-%d')} ({len(train_df):,} rows)")
print(f"Validation Set Horizon: {val_df['date'].min().strftime('%Y-%m-%d')} to {val_df['date'].max().strftime('%Y-%m-%d')} ({len(val_df):,} rows)")


## EDA

Many item-days have zero recorded sales. How common are zeros in this training subset?

In [ ]:
eda = train_df.copy()
eda["day_of_week"] = eda["date"].dt.dayofweek          # Monday = 0
eda["month_label"] = eda["date"].dt.to_period("M").astype(str)
eda["has_event"] = eda[["event_name_1", "event_name_2"]].notna().any(axis=1)
print("eda df created.")

In [ ]:
zero_share = eda["sales"].eq(0).mean()
positive = eda.loc[eda["sales"] > 0, "sales"]

print(f"Item-days in training period : {len(eda):,}")
print(f"Zero-sales item-days         : {zero_share:.1%}")
print(f"Mean units over all days     : {eda['sales'].mean():.3f}")
print(f"Median units over all days   : {eda['sales'].median():.0f}")
print(f"Mean units on selling days   : {positive.mean():.3f}")
print(f"Skewness of daily units      : {eda['sales'].skew():.2f}")
print(f"99th percentile              : {eda['sales'].quantile(0.99):.0f}")
print(f"Largest single observation   : {eda['sales'].max():.0f}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
eda["sales"].eq(0).value_counts().reindex([True, False], fill_value=0) \
    .set_axis(["Zero", "Positive"]).plot.bar(ax=axes[0], rot=0, color=["#b0b0b0", "#4c72b0"])
axes[0].set(title="Figure 1a. Zero vs positive demand (training days)",
            xlabel="Recorded demand", ylabel="Number of item-days")
axes[1].hist(np.log1p(positive), bins=40, color="#4c72b0")
axes[1].set(title="Figure 1b. Positive demand, log scale",
            xlabel="log(1 + units sold)", ylabel="Number of item-days")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "fig1_zero_and_positive_demand.png", dpi=150)
plt.show()

Individual items are too noisy to read. The trailing 7-day average smooths daily noise. It is a description of the past, not a forecast.

In [ ]:
daily = eda.groupby("date")["sales"].sum().sort_index()

fig, ax = plt.subplots(figsize=(13, 4))
daily.plot(ax=ax, alpha=0.45, label="Daily units sold")
daily.rolling(7).mean().plot(ax=ax, linewidth=2, label="Trailing 7-day average")
ax.set(title=f"Figure 2. CA_1 total daily units sold, training period",
       xlabel="Date", ylabel="Units sold per day")
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "fig2_daily_totals.png", dpi=150)
plt.show()

display(daily.describe().to_frame("store daily units"))
print("\n--- the five lowest days ---")
display(daily.nsmallest(5).to_frame("units"))

Weekly and monthly patterns

In [ ]:
daily_frame = daily.rename("units").reset_index()
daily_frame["weekday"] = daily_frame["date"].dt.dayofweek
daily_frame["month_label"] = daily_frame["date"].dt.to_period("M").astype(str)

monthly = daily_frame.groupby("month_label").agg(mean_daily_units=("units", "mean"),
                                                 observed_days=("units", "size"))
display(monthly.round(1))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.boxplot(data=daily_frame, x="weekday", y="units", ax=axes[0], color="#4c72b0")
axes[0].set_xticks(range(7))
axes[0].set_xticklabels(["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"])
axes[0].set(title="Figure 3a. Daily store totals by weekday", xlabel="", ylabel="Units sold per day")
monthly["mean_daily_units"].plot.bar(ax=axes[1], color="#4c72b0", rot=45)
axes[1].set(title="Figure 3b. Mean daily units by month", xlabel="Month", ylabel="Units sold per day")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "fig3_weekday_monthly.png", dpi=150)
plt.show()

print("Daily totals per weekday (n = number of days observed):")
display(daily_frame.groupby("weekday")["units"].agg(["size", "mean", "median"]).round(1))

Categories and departments

In [ ]:
category_summary = eda.groupby("cat_id").agg(total_units=("sales", "sum"),
                                             mean_units_per_item_day=("sales", "mean"),
                                             zero_share=("sales", lambda s: s.eq(0).mean()),
                                             items=("id", "nunique"))
display(category_summary.round(3))

dept_summary = eda.groupby("dept_id").agg(total_units=("sales", "sum"),
                                          mean_units_per_item_day=("sales", "mean"),
                                          items=("id", "nunique"))
display(dept_summary.round(3))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
category_summary["total_units"].sort_values().plot.barh(ax=axes[0], color="#4c72b0")
axes[0].set(title="Figure 4a. Total units by category", xlabel="Total units sold", ylabel="Category")
dept_summary["mean_units_per_item_day"].sort_values().plot.barh(ax=axes[1], color="#4c72b0")
axes[1].set(title="Figure 4b. Mean units per item-day by department",
            xlabel="Units per item-day", ylabel="Department")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "fig4_category_department.png", dpi=150)
plt.show()

Individual items and how intermittent they are

In [ ]:
item_summary = eda.groupby("id").agg(total_units=("sales", "sum"),
                                     mean_units=("sales", "mean"),
                                     zero_share=("sales", lambda s: s.eq(0).mean()))
print("--- top 10 items by training volume ---")
display(item_summary.sort_values("total_units", ascending=False).head(10).round(3))

TOP_ID = item_summary["total_units"].idxmax()
INTERMITTENT_ID = item_summary.loc[item_summary["total_units"] > 0, "zero_share"].idxmax()
print("High-volume example :", TOP_ID)
print("Intermittent example:", INTERMITTENT_ID)

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
for ax, item in zip(axes, [TOP_ID, INTERMITTENT_ID]):
    part = eda.loc[eda["id"].eq(item)]
    ax.plot(part["date"], part["sales"], linewidth=1, color="#4c72b0")
    ax.set(title=f"{item} (zero on {item_summary.loc[item, 'zero_share']:.0%} of training days)",
           ylabel="Units sold per day")
axes[-1].set_xlabel("Date")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "fig5_item_examples.png", dpi=150)
plt.show()

fig, ax = plt.subplots(figsize=(8, 3.5))
item_summary["zero_share"].plot.hist(bins=25, ax=ax, color="#4c72b0")
ax.set(title="Figure 6. How intermittent are the items?",
       xlabel="Share of training days with zero recorded sales", ylabel="Number of items")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "fig6_intermittency.png", dpi=150)
plt.show()

Missing Prices: what do those rows have in common?

In [ ]:
price_check = eda.assign(price_missing=eda["sell_price"].isna()).groupby("price_missing").agg(
    item_days=("sales", "size"),
    total_units=("sales", "sum"),
    mean_units=("sales", "mean"),
    max_units=("sales", "max"),
    zero_share=("sales", lambda s: s.eq(0).mean()))
display(price_check.round(4))

share_missing = eda["sell_price"].isna().mean()
print(f"\nShare of training item-days with no price: {share_missing:.1%}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
weekly_prices = eda[["item_id", "wm_yr_wk", "sell_price"]].drop_duplicates()
weekly_prices["sell_price"].dropna().plot.hist(bins=50, ax=axes[0], color="#4c72b0")
axes[0].set(title="Figure 7a. Observed weekly selling prices",
            xlabel="Selling price (USD)", ylabel="Number of item-weeks")
eda.groupby("date")["sell_price"].apply(lambda s: s.isna().mean()).plot(ax=axes[1], color="#4c72b0")
axes[1].set(title="Figure 7b. Share of items with no price, over time",
            xlabel="Date", ylabel="Fraction of items")
axes[1].set_ylim(0, 1)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "fig7_prices.png", dpi=150)
plt.show()
print("One row per item-week is used for the histogram, so a weekly price is not counted seven times.")

Price and sales within a single item

In [ ]:
one_item = eda.loc[eda["id"].eq(TOP_ID)]
item_weeks = one_item.groupby("wm_yr_wk").agg(price=("sell_price", "first"),
                                              mean_daily_sales=("sales", "mean"),
                                              observed_days=("sales", "size"))
display(item_weeks.head(8).round(3))
print("Distinct observed prices for this item:", item_weeks["price"].nunique())

fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(item_weeks["price"], item_weeks["mean_daily_sales"], alpha=0.7, color="#4c72b0")
ax.set(title=f"Figure 8. Weekly price vs mean daily units: {TOP_ID}",
       xlabel="Weekly selling price (USD)", ylabel="Mean units sold per day that week")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "fig8_price_vs_sales.png", dpi=150)
plt.show()
if item_weeks["price"].nunique() < 4:
    print("This item barely changed price, so the plot cannot say much about price response.")

Event days and SNAP days

In [ ]:
date_labels = eda[["date", "has_event", "snap_CA"]].drop_duplicates("date")
calendar_daily = daily_frame.merge(date_labels, on="date", validate="one_to_one")

for flag, label in [("has_event", "Named calendar event"), ("snap_CA", "California SNAP day")]:
    print(f"--- {label} ---")
    display(calendar_daily.groupby(flag).agg(days=("units", "size"),
                                             mean_units=("units", "mean"),
                                             median_units=("units", "median")).round(1))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.boxplot(data=calendar_daily, x="has_event", y="units", ax=axes[0], color="#4c72b0")
axes[0].set(title="Figure 9a. Event vs non-event days", xlabel="Named event that day",
            ylabel="Store units sold per day")
sns.boxplot(data=calendar_daily, x="snap_CA", y="units", ax=axes[1], color="#4c72b0")
axes[1].set(title="Figure 9b. SNAP vs non-SNAP days", xlabel="SNAP purchases allowed (CA)",
            ylabel="Store units sold per day")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "fig9_events_snap.png", dpi=150)
plt.show()

Correlations

In [ ]:
corr_cols = ["sales", "sell_price", "day_of_week"]
corr = eda[corr_cols].corr()
fig, ax = plt.subplots(figsize=(5.5, 4.5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, vmin=-1, vmax=1, ax=ax)
ax.set_title("Figure 10. Correlations, training period only")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "fig10_correlation.png", dpi=150)
plt.show()
print("Correlation is not causation, and pooling items of different scales inflates or masks it.")

## Feature preparation

Here we build features on the ordered history before splitting the feature rows. A 28-day lag ensures every validation lag and rolling window ends on or before December 3. Validation sales never enter validation features.

The 7-day and 30-day rolling summaries operate on `lag_28`. The first 57 days per series do not have a complete 30-day summary. Calendar features are known at the forecast origin. Prices are excluded from the candidate model inputs because future price availability has not been established. This section prepares inputs for a later ML comparator; it does not fit one.

In [ ]:
LAG_DAYS = 28
ROLLING_WINDOWS = (7, 30)
NUMERIC_FEATURES = ["lag_28", "rolling_mean_7", "rolling_mean_30", "rolling_std_7",
                    "day_of_week", "day_of_month", "month", "snap_CA"]
CATEGORICAL_FEATURES = ["item_id", "dept_id", "cat_id", "event_name_1", "event_type_1"]
MODEL_FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES

def build_features(history):
    # Limit the feature frame to columns needed by the later model.
    cols = list(dict.fromkeys(["id", "date", "sales", "month", "snap_CA"] + CATEGORICAL_FEATURES))
    out = history[cols].sort_values(["id", "date"]).reset_index(drop=True).copy()
    out["day_of_week"] = out["date"].dt.dayofweek
    out["day_of_month"] = out["date"].dt.day
    out["lag_28"] = out.groupby("id", observed=True)["sales"].shift(LAG_DAYS)
    for window in ROLLING_WINDOWS:
        out[f"rolling_mean_{window}"] = out.groupby("id", observed=True)["lag_28"].transform(
            lambda values: values.rolling(window, min_periods=window).mean())
    out["rolling_std_7"] = out.groupby("id", observed=True)["lag_28"].transform(
        lambda values: values.rolling(7, min_periods=7).std())
    return out.dropna(subset=NUMERIC_FEATURES).copy()

feature_frame = build_features(df)
train_features = feature_frame.loc[feature_frame["date"] <= split_date].copy()
val_features = feature_frame.loc[feature_frame["date"] > split_date].copy()
assert len(val_features) == len(val_df) == 3049 * 28
assert val_features.groupby("id").size().eq(28).all()
assert (val_features["date"] - pd.Timedelta(days=LAG_DAYS)).max() <= split_date

# Learn each vocabulary once, using training data only, and reuse it in validation.
# Unseen validation categories map to -1; the later model must handle that explicitly.
category_vocabularies = {}
for col in CATEGORICAL_FEATURES:
    train_values = train_features[col].fillna("__NO_EVENT__").astype(str)
    val_values = val_features[col].fillna("__NO_EVENT__").astype(str)
    vocabulary = sorted(train_values.unique().tolist())
    category_vocabularies[col] = vocabulary
    dtype = pd.CategoricalDtype(categories=vocabulary)
    train_features[col] = train_values.astype(dtype).cat.codes
    val_features[col] = val_values.astype(dtype).cat.codes

print(f"Usable training feature rows: {len(train_features):,}")
print(f"Usable validation feature rows: {len(val_features):,}")
print("Missing numeric validation features:", int(val_features[NUMERIC_FEATURES].isna().sum().sum()))
print("Unknown category counts in validation:")
display(val_features[CATEGORICAL_FEATURES].eq(-1).sum().to_frame("rows"))
with (OUTPUT_DIR / "feature_category_vocabularies.json").open("w") as handle:
    json.dump(category_vocabularies, handle, indent=2)
del feature_frame

## Baseline definitions

All methods predict the same 3,049 series for the same 28 days, without observing sales within the validation window.

| Method | Forecast at the December 3 origin |
|---|---|
| Last observation | Repeat December 3 sales for all 28 days. |
| Seasonal naive (7 days) | Repeat the final seven training days four times. |
| Moving average (28 days) | Repeat the mean of the final 28 training days. The window is specified in advance, not tuned on validation. |

Forecasts remain fractional when appropriate and are clipped to zero consistently. These are daily unit-sales forecasts, not inventory order quantities.

In [ ]:
train_matrix = train_df.pivot(index="id", columns="date", values="sales").sort_index().sort_index(axis=1)
val_matrix = val_df.pivot(index="id", columns="date", values="sales").reindex(train_matrix.index).sort_index(axis=1)
assert train_matrix.notna().all().all() and val_matrix.notna().all().all()
assert train_matrix.shape == (3049, 702)
assert val_matrix.shape == (3049, 28)
assert train_matrix.columns.max() == split_date
assert val_matrix.columns.min() == split_date + pd.Timedelta(days=1)
assert train_matrix.columns.equals(pd.date_range(train_matrix.columns.min(), split_date))
assert val_matrix.columns.equals(pd.date_range(split_date + pd.Timedelta(days=1), df["date"].max()))

def baseline_forecasts(history, horizon):
    history = np.asarray(history, dtype=float)
    assert history.ndim == 2 and history.shape[1] >= 28
    forecasts = {
        "Last observation": np.repeat(history[:, -1:], horizon, axis=1),
        "Seasonal naive (7d)": np.tile(history[:, -7:], (1, int(np.ceil(horizon / 7))))[:, :horizon],
        "Moving average (28d)": np.repeat(history[:, -28:].mean(axis=1, keepdims=True), horizon, axis=1),
    }
    return {name: np.maximum(values, 0.0) for name, values in forecasts.items()}

predictions = baseline_forecasts(train_matrix.to_numpy(), val_matrix.shape[1])
actual = val_matrix.to_numpy(dtype=float)
for name, predicted in predictions.items():
    assert predicted.shape == actual.shape
    assert np.isfinite(predicted).all() and (predicted >= 0).all()
print(f"Forecast origin: {split_date.date()}")
print(f"Forecasts per method: {actual.size:,} ({actual.shape[0]:,} series x {actual.shape[1]} days)")

## Forecast error metrics

For each series, MAE is the mean absolute validation error and RMSE is the square root of mean squared validation error. RMSSE divides validation MSE by the mean squared one-day sales change across **all 702 training days** before taking the square root. This explicit scaling convention includes leading zeros.

A zero training scale makes RMSSE undefined. Those series remain in MAE and RMSE and are listed separately for RMSSE. The main summary averages per-series scores with equal weight. Pooled RMSE is also shown separately and is not the same as mean per-series RMSE. No volume weights or M5 hierarchy are used, so these are not official M5 competition scores. MAPE is omitted because actual sales can be zero.

In [ ]:
series_info = train_df[["id", "item_id", "cat_id", "dept_id"]].drop_duplicates("id").set_index("id").reindex(train_matrix.index)
scale = np.mean(np.diff(train_matrix.to_numpy(dtype=float), axis=1) ** 2, axis=1)
scalable = scale > 0
unscalable = series_info.loc[~scalable].copy()
unscalable["training_mean_sales"] = train_matrix.loc[~scalable].mean(axis=1)
unscalable.to_csv(OUTPUT_DIR / "rmsse_unscalable_series.csv")

score_parts = []
for name, predicted in predictions.items():
    error = predicted - actual
    mse = np.mean(error ** 2, axis=1)
    rmsse = np.full(len(scale), np.nan)
    rmsse[scalable] = np.sqrt(mse[scalable] / scale[scalable])
    scores = series_info.copy()
    scores["model"] = name
    scores["MAE"] = np.mean(np.abs(error), axis=1)
    scores["RMSE"] = np.sqrt(mse)
    scores["RMSSE"] = rmsse
    scores["training_scale"] = scale
    score_parts.append(scores.reset_index())
per_series_scores = pd.concat(score_parts, ignore_index=True)
summary = per_series_scores.groupby("model", sort=False).agg(
    series=("id", "size"), mean_MAE=("MAE", "mean"),
    mean_RMSE=("RMSE", "mean"), mean_RMSSE=("RMSSE", "mean"),
    RMSSE_series=("RMSSE", "count"))
summary["pooled_RMSE"] = [np.sqrt(np.mean((predictions[name] - actual) ** 2)) for name in summary.index]
print("Unweighted mean per-series validation scores (lower is better):")
display(summary.round(4))
print(f"RMSSE unavailable for {len(unscalable):,} training-constant series.")
if len(unscalable):
    display(unscalable.head(10))
per_series_scores.to_csv(OUTPUT_DIR / "baseline_per_series_metrics.csv", index=False)
summary.to_csv(OUTPUT_DIR / "baseline_summary.csv")

category_scores = per_series_scores.groupby(["cat_id", "model"], sort=False).agg(
    series=("id", "size"), mean_MAE=("MAE", "mean"), mean_RMSE=("RMSE", "mean"),
    mean_RMSSE=("RMSSE", "mean"), RMSSE_series=("RMSSE", "count"))
print("Scores by product category (categories are not demand regimes):")
display(category_scores.round(4))
category_scores.to_csv(OUTPUT_DIR / "baseline_category_metrics.csv")

quantiles = per_series_scores.groupby("model", sort=False)[["MAE", "RMSE", "RMSSE"]].quantile([0.25, 0.5, 0.75, 0.9])
print("Per-series score distributions:")
display(quantiles.round(4))
quantiles.to_csv(OUTPUT_DIR / "baseline_score_quantiles.csv")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, metric, title in zip(axes, ["mean_MAE", "mean_RMSE", "mean_RMSSE"],
                            ["Mean per-series MAE", "Mean per-series RMSE", "Mean per-series RMSSE"]):
    summary[metric].plot.bar(ax=ax, color=["#8994a8", "#0f7f80", "#e7a23a"], rot=25)
    ax.set(title=title, xlabel="", ylabel="Error (lower is better)")
fig.suptitle("Fixed-origin validation: December 4–31, 2015")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "fig11_baseline_scores.png", dpi=150)
plt.show()

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(val_matrix.columns, actual.sum(axis=0), color="black", linewidth=2, label="Recorded sales")
for name, predicted in predictions.items():
    ax.plot(val_matrix.columns, predicted.sum(axis=0), label=name, alpha=0.8)
ax.set(title="Store totals: descriptive view of the same validation forecasts",
       xlabel="Validation date", ylabel="Units per day")
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "fig12_baseline_store_totals.png", dpi=150)
plt.show()

## Checks and reproducible outputs

The checks below verify baseline horizon alignment and demonstrate that changing held-out sales cannot change validation features. They also verify the feature rows and saved forecast dimensions. Results cover a single December validation window and do not establish performance in other seasons or inventory savings.

In [ ]:
# Known-history checks catch seasonal alignment and moving-average mistakes.
toy_history = np.arange(1, 29, dtype=float)[None, :]
toy = baseline_forecasts(toy_history, 28)
np.testing.assert_array_equal(toy["Last observation"], np.full((1, 28), 28.0))
np.testing.assert_array_equal(toy["Seasonal naive (7d)"], np.tile(np.arange(22, 29), (1, 4)))
np.testing.assert_allclose(toy["Moving average (28d)"], 14.5)

# Perturb validation sales for three complete series; predictors must be unchanged.
check_ids = train_matrix.index[:3]
check_history = df.loc[df["id"].isin(check_ids)].copy()
changed_history = check_history.copy()
changed_history.loc[changed_history["date"] > split_date, "sales"] = 999999
original_features = build_features(check_history)
changed_features = build_features(changed_history)
pd.testing.assert_frame_equal(
    original_features.loc[original_features["date"] > split_date, ["id", "date"] + MODEL_FEATURES].reset_index(drop=True),
    changed_features.loc[changed_features["date"] > split_date, ["id", "date"] + MODEL_FEATURES].reset_index(drop=True))
assert len(per_series_scores) == 3049 * 3
assert per_series_scores[["MAE", "RMSE"]].notna().all().all()
assert per_series_scores["RMSSE"].isna().sum() == len(unscalable) * 3

# Save predictions in a compact wide table, one row per item and validation day.
forecast_rows = pd.DataFrame({
    "id": np.repeat(val_matrix.index.to_numpy(), actual.shape[1]),
    "date": np.tile(val_matrix.columns.to_numpy(), actual.shape[0]),
    "actual_sales": actual.ravel(),
})
for name, predicted in predictions.items():
    forecast_rows[name] = predicted.ravel()
forecast_rows.to_csv(OUTPUT_DIR / "baseline_validation_predictions.csv.gz", index=False, compression="gzip")
protocol = {
    "train_start": str(train_matrix.columns.min().date()), "forecast_origin": str(split_date.date()),
    "validation_start": str(val_matrix.columns.min().date()), "validation_end": str(val_matrix.columns.max().date()),
    "protocol": "fixed origin, no validation updates", "series": len(train_matrix),
    "validation_days": actual.shape[1], "moving_average_window": 28,
    "rmsse_scale": "mean squared one-day change over all training observations, including leading zeros",
    "rmsse_unscalable_series": len(unscalable), "final_test_evaluated": False,
    "regimes_classified": False,
    "input_sha256": hashlib.sha256((DATA_DIR / "m5_ca1_2014_2015_prepared.csv.gz").read_bytes()).hexdigest(),
}
(OUTPUT_DIR / "baseline_protocol.json").write_text(json.dumps(protocol, indent=2))
print("All baseline and feature checks passed.")
print("Outputs:", OUTPUT_DIR.resolve())
for metric in ["mean_MAE", "mean_RMSE", "mean_RMSSE"]:
    print(f"Lowest {metric}: {summary[metric].idxmin()} ({summary[metric].min():.4f})")

## Interpretation and next steps

**Result:** The 28-day moving average has the lowest mean per-series MAE (1.0887), RMSE (1.4016), and RMSSE (0.7234) among the three baselines. RMSSE covers 3,043 series; six training-constant series have an undefined scale and remain included in MAE and RMSE.

The displayed rankings apply to this fixed-origin validation window and the stated metric aggregation. A seasonal pattern in training data does not guarantee that the seasonal-naive method wins. SNAP and event comparisons are descriptive associations. Zero sales on December 25 do not independently establish a store closure.

**Completed here:** training-only EDA, corrected feature preparation, three baseline forecasts, per-series and category-level validation scores, and leakage checks.

**Still pending:** training-based demand-regime rules, the pooled LightGBM comparator, additional chronological validation origins, an independent final test period, and the inventory simulation. Any later model selection should use validation only. Document the final test dates and keep its outcomes untouched until the method choices are fixed. Inventory cost and service improvements remain research questions.